$ TBIA爬蟲找植物名錄、植物物種紀錄 $


目的：
為了製作物種分布模擬(species distribution model, SDM)的生物出現紀錄圖層，我們需要有物種的出現-未出現資料(presence-absence data)。由於調查方法與時效性限制，出現紀錄未能涵蓋該物種在樣區所布集的出現位置與出現時間。因此，我們從台灣生物多樣性資訊聯盟(Taiwan biodiversity information association, TBIA)上下載蒐集研究性的物種出現紀錄，用於在後續使用拔靴法(Bootstrapping)隨機抽取資料時，有足夠多的樣本數。在後續機器學習訓練模型時，期能增加模型母數N的可信度，使本研究結果能更貼切地描述物種所處的生態棲位。


方法：
按照excel名單的物種資料(先對植物分群)產生名錄，按名錄順序上網搜尋物種資料。蒐集到的物種的學名、分類位階、紀錄筆數、紀錄時間、紀錄座標 (含模糊化處理)的資料依序排好，並寫入新的excel檔。


流程表：
1.  安裝爬蟲模組，並初始化TBIA測試。
    - get_TBIA_html(url)    =>    soup：
    - get_current_url(x_path)    =>    new_url：

2.  按設定的類群分批產生物種名錄。
    - mk_name_list(raw_file, type_)    =>    name_list：
        將輸入的excel檔名，先選擇對植物分群的參數，後轉成植物由俗名構成的暫時名錄，將俗名輸出成list。
    - mk_url_list(name_list)    =>    return url_list：
        對暫時名錄的中文名進行utf-8編碼，將編碼與TBIA物種搜尋網頁結合，製作成TBIA物種搜尋結果的網址，後將TBIA搜尋網址輸出成list。

3.  提取TBIA物種搜尋網頁資料(第一階)。
    由於第一階的網頁html檔所包含的內容有限，僅能得到物種學名、物種介紹連結、物種出現紀錄連結。但能提供超連結網頁，透過加載超連結網址，在第二階的網頁搜尋中取得物種分類位階、物種出現紀錄等資料。
    - get_sp_basic_url(title)    =>    sp_basic_url：
    - get_sp_scinane(title)    =>    verName, sciName：
    - get_sp_data_url(title)    =>    sp_occur_record_url, sp_nature_history_url：
    - quit_crawler    =>    是否退出爬蟲：
    - 產生for迴圈，遍歷TBIA俗名搜尋，帶入上述函數。
    - for_1st_loop(title)    =>    vername, sciname, sp_data_url, sp_occur_record_url, sp_nature_history：

4.  寫入txt儲存第一階爬蟲結果
    將第一階的成果寫入txt檔暫存，以備當網路斷訊、當機、不當終止執行，等造成cpu不再暫存爬蟲的搜尋結果後，不必再花時間重新爬蟲，即可隨時方便地取得並收 藏爬蟲的結果。

1. write_read_txt(file_name, new_content)、read_dict_from_txt(file_name)、read_file_by_lines(file_name)、write_read_txt(file_name, new_content)、save_excel_file(file_name)、quit_crawler()

In [ ]:
# 安裝基礎模組、靜態爬蟲模組、建立隨機時間間隔
import requests as req
from bs4 import BeautifulSoup as BS
# import pandas as pd

import files
import json
import openpyxl as opx
import os

import re
import math
import random
import time
import threading

delay_times = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.0, 2.0]    # 延遲的秒數
delay = random.choice(delay_times)    # 隨機選取秒數

In [3]:
# 安裝動態爬蟲模組selenium
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service as ChromeService

Options().add_argument('--headless')
Options().add_argument('--no-sandbox')
Options().add_argument('--disable-dev-shm-usage')
Options().add_argument('--disable-blink-features=AutomationControlled')


# 使用Cookie偽裝成使用者
Options().add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36')
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
driver.quit()

In [4]:
# 將內容逐行寫入記事本內，搭配迴圈使用，每次打開寫入內容
def write_read_txt(file_name, new_content):
    try:
        with open(file_name, 'r', encoding ='utf-8') as file:       # read模式，回傳內容
            old_content = file.read()
    except FileNotFoundError:
        old_content = ''
    
    if new_content is not None:
        with open(file_name, 'a', encoding ='utf-8') as file:       # append模式，避免write覆寫掉舊內容
            file.write(str(new_content) +'\n')
            file.close()
    return old_content


# 將儲存在記事本的資料寫入新excel
def read_dict_from_txt(file_name):
    with open(file_name, 'r', encoding ='utf-8') as file:
        dict_list = []
        for line in file:
            try:
                dict_obj = json.loads(line.strip())  # 將每行的字串轉成字典
                dict_list.append(dict_obj)
            except json.JSONDecodeError:
                print(f"解析失敗，跳過此行: {line}")
    return dict_list

In [5]:
# 逐行讀取txt檔內的資料，每項資料一行
def read_file_by_lines(file_name):
    try:
        with open(file_name, 'r', encoding='utf-8') as file:
            for line in file:
                # 使用 strip() 去除行首尾的空格和換行符號
                yield line.strip() + ','  # 使用 yield 生成每一行
    except FileNotFoundError:
        print(f'檔案 {file_name} 不存在')


# 將txt檔的內容類型變成string
def mk_txt_str(file_name):
    content = read_file_by_lines(file_name)
    content = ''.join(content)

    txt_str = content.replace('\n', '')  # 移除換行符號
    txt_str = txt_str.replace("'", '"')  # 將單引號替換為雙引號
    txt_str = txt_str.replace("{", "[")  # 替換大括號為中括號
    txt_str = txt_str.replace("}", "]")  # 替換大括號為中括號
    txt_str = f"[{txt_str.strip()}]"  # 確保資料包裝在 JSON 陣列中
    return txt_str

In [6]:
# 開啟新excel檔
def excel_active(sheet_title):
    wb = opx.Workbook()
    ws = wb.active
    ws.title = sheet_title
    return ws

# 儲存excel檔並下載
def save_excel_file(file_name):
    wb = opx.Workbook()
    wb.save(file_name)
    # files.download(file_name)

In [7]:
# 如果quit_crawler()的input在3秒內按q，則跳出迴圈，否則就跳過quit_crawler()，繼續執行迴圈
def quit_crawler():
    def wait():
        quit = threading.Event()
        _str = '按q鍵+Enter可終止爬蟲程式：\n'
        input_str = input(_str).strip().lower()
        if input_str == 'q':
            quit.set()

    thread = threading.Thread(target=wait, daemon=True)
    thread.start()
    thread.join(timeout=3)

    if quit.is_set():
        print('退出爬蟲')
        return True
    else:
        print('未输入q，继续执行爬蟲\n------------------------------')
        return False

2. req_TBIA_html(url)、drive_TBIA_html(url)、get_current_url_attr(url, By, attribute)、get_current_url_link(url, link)、mk_name_list(rawData, type_)、mk_search_url(title)、mk_reduc_title(url)、search_test(title, search_url)

In [8]:
# 取得url網站的html格式
# TBIA網站有相應的User-Agent

# 靜態
def req_TBIA_html(url):
    # 使用Cookie偽裝成使用者
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'}
    response = req.get(url, headers=headers)
    # print(response.text)

    # 進入網頁
    if response.status_code == 200:
        pass
    else:
        print(f'Request failed with status code: {response.status_code}')
        return None
    soup = BS(response.text, 'html.parser')
    # print(soup.prettify())
    return soup


# 動態
def drive_TBIA_html(url):
    # 使用Cookie偽裝成使用者
    Options().add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36')
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    driver.get(url=url)    # 進入網站，url=url
    # print(driver.page_source)

    # 進入網頁
    req_TBIA_html(url)
    soup = BS(driver.page_source, 'html.parser')
    # print(soup.prettify())
    # driver.quit()
    return soup

In [9]:
# 取得點擊html的element中，具有超連結功能的網頁網址
def get_current_url_attr(url, By, attribute):
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    driver.get(url=url)    # 進入網站，url=url
    # action = webdriver.ActionChains(driver)
    ''''''
    link = driver.find_element(By, attribute)
    link.click()
    time.sleep(delay)
    new_url = driver.current_url
    ''''''
    # driver.quit()
    return new_url


def get_current_url_link(url, link):
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    driver.get(url=url)    # 進入網站，url=url
    # action = webdriver.ActionChains(driver)
    ''''''
    if link:
        link.click()
    else:
        print('未找到可點擊的element')
        return None
    time.sleep(delay)
    new_url = driver.current_url
    ''''''
    # driver.quit()
    return new_url

In [10]:
# 在此輸入excel檔名
# 物種的俗名名錄必須在檔案的第3欄、物種類型篩選必需在檔案的第5欄，且檔案必須為xlsx格式
'''合歡山公路調查植物表..xlsx'''
raw_file = '合歡山公路調查植物表..xlsx'

# 待搜尋的物種名稱與其類群屬性
title_col = 'Unnamed: 2'
typen_col = 'Unnamed: 4'

# 將excel上的物種俗名名錄轉成列表
# 若type_為''，則不區分植物類型，若type_為數字，則只取出該相應類型的植物
def mk_name_list(rawData, type_):
    rawData = pd.read_excel('合歡山公路調查植物表..xlsx')
    rawData = pd.DataFrame(rawData)

    name_list = []
    for i in range(len(rawData)):
        title = rawData[title_col][i]
        typen = rawData[typen_col][i]

        # ''：不區分植物類型；numeric(1~13)：只取出該相應類型的植物
        if (type_ != '') and (type_ != typen):
            continue
        # 清理不符合條件的物種名稱欄位
        if (pd.notna(title) == False) or (title == 'index') or any(keyword in title for keyword in ['科', '屬', 'sp.']):
            continue
        title = str(title)
        typen = int(math.floor(typen))          if typen is not None and not math.isnan(typen) else None

        name_list.append([title, typen])
    return name_list

In [11]:
# 用UTF-8編碼，將Unicode中文字解碼成相對應的百分比編碼加碼
# 16進位，每1個中文字元由9個字元組，其中有3組%、6個16進位數字
from urllib.parse import quote, unquote

def mk_search_url(title):
    url_input = 'https://tbiadata.tw/zh-hant/search/full?keyword='

    # 製作搜尋物種的網址，並製作與之相對應搜尋清單的字典
    decode_text = str(title)
    encode_text = quote(decode_text, encoding='utf-8')
    search_url = url_input + encode_text
    return search_url

def mk_reduc_title(url):
    encode_text = url.split('=')[-1]
    decode_text = unquote(encode_text, encoding='utf-8')
    title = decode_text
    return title

In [12]:
# 製作可供TBIA搜尋的列表，使後續爬蟲順利載入且具有一致化格式
def search_test(title, search_url):
    search_url = mk_search_url(title)

    # 檢視TBIA是否能搜尋到該物種
    soup = req_TBIA_html(search_url)
    sp_item = soup.find('div', class_='item item_spe')
    title, search_url = None, None
    if sp_item.find('div', class_='title').find('p').text == '物種 (0)':
        print(f'搜尋 {title} 失敗，可能為拼字錯誤')
        return
    elif sp_item.find('div', class_='no_data'):
        if sp_item.find('div', class_='no_data').text == '無資料':
            print(f'搜尋 {title} 失敗，可能為拼字錯誤')
            return
    
    # 檢視TBIA搜尋到該物種是否搜尋到其他有包含title的關鍵字
    if sp_item.find('span', class_='col_red').text != title:
        return
    return title, search_url

3. get_sciname__found_url(url)、get_taxon_url(url)、mk_taxon_rank(sp_taxon_url)、get_occur_record_url(url)、get_nature_history_url(url)

In [13]:
# 進入TBIA的物種搜尋網頁，輸入物種俗名(vernacular name)提取物種學名(scientific name)
# 尋找element，提取介紹物種學名
def get_sciname__found_url(url):
    title = mk_reduc_title(url)
    soup = req_TBIA_html(url)
    sp_item = soup.find('div', class_='item item_spe')
    sp_item = soup.find('div', class_='flex_top')

    if (not sp_item) or (not sp_item.find('i')):
        return title, str(None), str(None)

    verName = sp_item.find('span', class_='col_red')
    sciName = sp_item.find('i')
    if verName or sciName:
        verName = verName.text  # 提取text內文
        sciName = sciName.text  # 提取text內文
        sp_found_url = url

    # print(f'提取 {title} 學名:  ' + sciName)
    return verName, sciName, sp_found_url

In [14]:
# 進入TBIA物種搜尋網頁，提取物種資料的網址
def get_taxon_url(url):
    title = mk_reduc_title(url)
    soup = req_TBIA_html(url)
    sp_item = soup.find('div', class_='item item_spe')

    sp_item_all = sp_item.find_all('li')
    sciname = get_sciname__found_url(url)[1]

    sp_taxon_url = None
    for i in range(len(sp_item_all)):
        sp_item = sp_item_all[i]
        if (str(sp_item.find('i').text) != str(sciname)) or ('：種' not in sp_item.text):
            continue
        if (not sp_item) or (sp_item.find('div', class_='no_data')):
            continue

        link = sp_item.find('div', class_='btn_area')
        link = link.find_all('a', target='_blank')          if link else []
        if len(link) < 2:          # 確保列表中至少有2個元素
            continue
        '''# link = link[0]    # 物種介紹'''
        link = link[1]    # 台灣物種名錄
        sp_taxon_url = link['href']    # 提取href屬性
        break

    # print(f'提取 {title} 分類位階網址:  ' + sp_taxon_url)
    return sp_taxon_url

In [15]:
# 搜尋台灣物種名錄，提取分類階層資訊
def mk_taxon_rank(sp_taxon_url):
    taxon_soup = req_TBIA_html(sp_taxon_url)
    taxon_items = taxon_soup.find('div', class_='rank-area')
    taxon_items = taxon_items.find_all('div', class_='item')

    # 製作分類階層
    for i in range(len(taxon_items)):
        item = taxon_items[i]

        if item.find('div', class_='cir-box rank-1-red'):
            kingdom = item.find('a', class_='rank-p')
            kingdom = kingdom.text.strip()      if kingdom else None
        elif item.find('div', class_='cir-box rank-2-org'):
            phylum = item.find('a', class_='rank-p')
            phylum = phylum.text.strip()        if phylum else None
        elif item.find('div', class_='cir-box rank-3-yell'):
            _class = item.find('a', class_='rank-p')
            _class = _class.text.strip()        if _class else None
        elif item.find('div', class_='cir-box rank-4-green'):
            order = item.find('a', class_='rank-p')
            order = order.text.strip()          if order else None
        elif item.find('div', class_='cir-box rank-5-blue'):
            family = item.find('a', class_='rank-p')
            family = family.text.strip()        if family else None
        elif item.find('div', class_='cir-box rank-6-deepblue'):
            genus = item.find('a', class_='rank-p')
            genus = genus.text.strip()          if genus else None
        else:
            continue
    taxon = { 'kindom': kingdom, 'phylum': phylum, 'class': _class, 'order': order, 'family': family, 'genus': genus }

    # print(f'提取分類階層資訊：  {kingdom} , {phylum} , {_class} , {order} , {family} , {genus}')
    return kingdom, phylum, _class, order, family, genus

In [16]:
# 物種出現紀錄(occur_record, ocr)
# 進入TBIA物種搜尋網頁，提取物種出現紀錄資料的網頁連結(含資料筆數)
def get_occur_record_url(url):
    title = mk_reduc_title(url)
    soup = req_TBIA_html(url)
    sp_ocr_ = soup.find('div', class_='item item_occ')

    sp_ocr_all = sp_ocr_.find_all('li', {'data-key': 'taxonID'})
    sciName = get_sciname__found_url(url)[1]

    sp_ocr_num, sp_ocr_url = None, None
    for i in range(len(sp_ocr_all)):
        sp_ocr = sp_ocr_all[i]
        i_tag = sp_ocr.find('i')
        if i_tag and str(i_tag.text) != str(sciName):
            continue
        if '：種' not in sp_ocr.text:
            continue

        if sp_ocr:
            sp_ocr_num = sp_ocr.find('div', class_='num').text
            sp_ocr_url = get_current_url_attr(url, By.XPATH, '//*[@id="item_occ"]/ul/li[1]')    # 取得目前所在網址

        # print(f'提取 {title} 出現紀錄網址，共含 {sp_ocr_num} 筆:  ' + sp_ocr_url)
    return sp_ocr_num, sp_ocr_url

In [17]:
# 自然史典藏(nature_history, nh)
# 進入TBIA物種搜尋網頁，提取物種出現紀錄資料的網頁連結(含資料筆數)
def get_nature_history_url(url):
    title = mk_reduc_title(url)
    soup = req_TBIA_html(url)
    sp_nh_ = soup.find('div', class_='item item_col')

    sp_nh_all = sp_nh_.find_all('li', {'data-key': 'taxonID'})
    sciName = get_sciname__found_url(url)[1]

    sp_nh_num, sp_nh_url = None, None
    for i in range(len(sp_nh_all)):
        sp_nh = sp_nh_all[i]
        i_tag = sp_nh.find('i')
        if i_tag and str(i_tag.text) != str(sciName):
            continue
        if '：種' not in sp_nh.text:
            continue

        if sp_nh:
            sp_nh_num = sp_nh.find('div', class_='num').text
            sp_nh_url = get_current_url_attr(url, By.XPATH, '//*[@id="item_col"]/ul/li[1]')    # 取得目前所在網址

    # print(f'提取 {title} 自然史典藏網址，共含 {sp_nh_num} 筆:  ' + sp_nh_url)
    return sp_nh_num, sp_nh_url

5. mk_search_list(name_list)、iter_TBIA_result(search_list)

In [18]:
# 選取植物的物種分類類別，使該類植物名錄被挑選出，放入迴圈內爬蟲
sorts = {
    '' : '全部植物', 
    1  : '喬木', 
    2  : '大灌木 (高>50cm)', 
    3  : '小灌木 (高<50cm)', 
    4  : '大型草本 (高>50 m)', 
    5  : '中型草本 (50cm>高>20cm)', 
    6  : '小型草本 (20cm>高>5cm，直徑>20cm)', 
    7  : '極小型草本 (高~3cm，直徑<20cm)', 
    8  : '外來種植物', 
    9  : '裸子植物', 
    10 : '蕨類植物', 
    11 : '禾本目植物', 
    12 : '單子葉植物 (非禾本目)', 
    13 : '其他植物' }

sort = input('輸入物種分類類別：  \n')
if sort == '':
    print(f'您所選取的物種分類類別為：  "{sort}" ， {sorts[sort]}')
    name_list = mk_name_list(raw_file, sort)
    print(name_list)
elif int(sort) in range(1,14):
    s = int(sort)
    print(f'您所選取的物種分類類別為：  "{sort}" ， {sorts[s]}')
    name_list = mk_name_list(raw_file, s)
    print(name_list)
else:
    print('您所輸入的代碼不在上列字典中，請重新執行程式碼，並輸入正確代碼')

您所選取的物種分類類別為：  "" ， 全部植物
[['紅檜', 9], ['刺柏', 9], ['玉山圓柏', 9], ['台灣冷杉', 9], ['台灣雲杉', 9], ['台灣華山松', None], ['台灣二葉松', 9], ['台灣鐵杉', 9], ['阿里山五味子', 2], ['霧社木薑子', None], ['玉山木薑子', None], ['霧社楨楠', None], ['風藤', 5], ['高山小檗', None], ['川上氏小檗', 2], ['玉山小檗', 3], ['南投小檗', None], ['玉山十大功勞', 3], ['彎果黃堇', 5], ['黃堇', 5], ['台灣烏頭', 4], ['匍枝銀蓮花', 5], ['小白頭翁', 4], ['小木通', 4], ['掌葉毛茛', None], ['蓬萊毛茛', 6], ['高山毛茛', 6], ['鹿場毛茛', 6], ['密葉唐松草', 6], ['傅氏唐松草', 6], ['台灣金蓮花', 5], ['阿里山清風藤', 2], ['昆欄樹', 1], ['穗花八寶', 7], ['星果佛甲草', 6], ['玉山佛甲草', 6], ['台灣茶藨子', 2], ['落新婦', 4], ['阿里山落新婦', 5], ['台灣嗩吶草', 6], ['刺果衛矛', 1], ['福建賽衛矛', None], ['梅花草', 6], ['佩羅特木', None], ['台灣山酢漿草', 6], ['玉山金絲桃', 6], ['短柄金絲桃', 7], ['褐毛柳', 1], ['高山柳', None], ['如意草', None], ['喜岩堇菜', 6], ['箭葉堇菜', 6], ['雙黃花堇菜', 6], ['紫花地丁', None], ['尖山堇菜', None], ['絞股藍', 6], ['豌豆', 5], ['黃菽草', 8], ['白花三葉草', 8], ['瓜子金', 7], ['鄧氏胡頹子', 1], ['小葉胡頹子', None], ['阿里山黃鱔藤', None], ['大黃鱔藤', None], ['塔山鼠李', None], ['小葉鼠李', None], ['畢祿山鼠李', 2], ['細尖栒子', None], ['泡葉栒子', None], ['矮生

In [19]:
# 將所篩選出的植物類別，剔除不合理、無資料的項目，後製作成search_list
def mk_search_list(name_list):
    search_list = []
    for i in range(len(name_list)):
        title = name_list[i][0]
        search_url = mk_search_url(title)
        typen = name_list[i][1]

        title, search_url = search_test(title, search_url)
        if title == None or search_url == None:
            continue
        search_list.append([title, search_url, typen])
        yield search_list
    print(search_list)

In [20]:
# 主要在執行爬蟲的程式碼，在迴圈中連線到TBIA，取得學名、分類地位、物種出現紀錄、自然典藏史
def iter_TBIA_result(search_list):
    for i in range(len(search_list)):
        title = search_list[i][0]
        search_url = search_list[i][1]
        typen = search_list[i][2]

        # 設定results，用於存取是否找到學名、分類地位、物種出現紀錄、自然典藏史等資料及網站的輸出值
        # 學名：get_sciname__found_url(url)、分類地位：get_taxon_url(url)、物種出現紀錄：get_occur_record_url(url)、自然典藏史：get_nature_history_url(url)
        verName, sciName, found_url = get_sciname__found_url(search_url)
        sp_taxon_url = get_taxon_url(search_url)
        if sp_taxon_url == None:
            continue
        taxon_rank = mk_taxon_rank(sp_taxon_url)
        [kingdom, phylum, _class, order, family, genus] = [ taxon_rank[0], taxon_rank[1], taxon_rank[2], taxon_rank[3], taxon_rank[4], taxon_rank[5] ]

        ocr_num, ocr_url = get_occur_record_url(search_url)
        if ocr_url == None:
            continue
        nh_num, nh_url = get_nature_history_url(search_url)
        if nh_url == None:
            continue
        driver.quit()

        print(f'提取 {verName} 學名:  {sciName}')
        print(f'提取 {verName} 分類位階:  {kingdom}, {phylum}, {_class}, {order}, {family}, {genus}')
        print(f'提取 {verName} 出現紀錄網址，共 {ocr_num} 筆:  {ocr_url}')
        print(f'提取 {verName} 自然史典藏網址，共 {nh_num} 筆:  {nh_url}')

        results = { 'verName': verName, 'sciName': sciName, 
                    'category': typen, 'found_url': found_url,
                    'taxon_rank_kingdom': kingdom, 'taxon_rank_phylum': phylum, 'taxon_rank_class': _class, 'taxon_rank_order': order, 'taxon_rank_family': family, 'taxon_rank_genus': genus, 
                    'occur_record_num': ocr_num, 'occur_record_url': ocr_url,
                    'nature_history_num': nh_num, 'nature_history_url': nh_url }

        results = results + ','
        output_sp_result = write_read_txt('output_sp_result.txt', results)
        yield output_sp_result

6. file_line(file_name)、mk_result_list(file_name, head, tail)、mk_output_excel(sheet_title, result_list_)

In [21]:
# 使用範例：對檔案每一行進行遍歷
def file_line(file_name):
    result = ''
    for line in read_file_by_lines(file_name):  # 調用函數來獲取每一行
        result = str(result) + str('\n' + line)
        # print(line)
    return result

In [22]:
# 將第一階爬蟲結果製作列表
def mk_result_list(file_name):
    result_str = mk_txt_str(file_name)
    rslt_list = list(result_str.split('],['))
    result_list = []
    for rslt in enumerate(rslt_list):
        rslt = str(rslt)
        rslt = rslt.split(',')
        rslt = list(rslt)
        item_list = []
        for item in rslt:
            item = item.split(',')
            item = list(item)
            kv_list = []
            for kv in item:
                kv = kv.split(':')
                kv = list(kv)
                kv_list.append(kv)
            item_list.append(kv_list)
        result_list.append(item_list)

    return result_list

In [23]:
# 將儲存在txt的資料寫入新excel
def mk_output_excel(sheet_title, txt_file_name):
    result_lists = mk_result_list(txt_file_name)
    ws = excel_active(sheet_title)

    ws.cell(row=1, column=1).value = 'occurrenceID'     #流水號
    ws.cell(row=1, column=2).value = 'vernacularName'   #俗名
    ws.cell(row=1, column=3).value = 'scientificName'   #學名
    ws.cell(row=1, column=4).value = 'verbatimCoordinateSystem'     #座標
    ws.cell(row=1, column=5).value = 'eventDate'    #日期
    ws.cell(row=1, column=6).value = 'category'     #類別
    ws.cell(row=1, column=7).value = 'found_url'    #搜尋網址
    ws.cell(row=1, column=8).value = 'kingdom'   #界
    ws.cell(row=1, column=9).value = 'phylum'    #門
    ws.cell(row=1, column=10).value = 'class'    #綱
    ws.cell(row=1, column=11).value = 'order'    #目
    ws.cell(row=1, column=12).value = 'family'   #科
    ws.cell(row=1, column=13).value = 'genus'    #屬
    ws.cell(row=1, column=14).value = 'occur_record_num'     #物種出現紀錄次數
    ws.cell(row=1, column=15).value = 'occur_record_url'     #物種出現紀錄網址
    ws.cell(row=1, column=16).value = 'nature_history_num'   #自然史典藏
    ws.cell(row=1, column=17).value = 'nature_history_url'   #自然史典藏


    a = 2
    for result_list in enumerate(result_lists, start=2):    
        try:
            # 依字典順序讀取字典中的學名、分類地位、物種出現紀錄、自然典藏史等資料及網站
            verName = result_list[1][1]
            sciName = result_list[2][1]
            category = result_list[3][1]
            found_url = result_list[4][1]
            kingdom = result_list[5][1]
            phylum = result_list[6][1]
            _class = result_list[7][1]
            order = result_list[8][1]
            family = result_list[9][1]
            genus = result_list[10][1]
            occur_record_num = result_list[11][1]
            occur_record_url = result_list[12][1]
            nature_history_num = result_list[13][1]
            nature_history_url = result_list[14][1]
        except IndexError as e:
            continue

        occurrenceID = a-1
        # 將字典中的學名、分類地位、物種出現紀錄、自然典藏史等資料及網站依序放入excel
        ws.cell(row=a, column=2).value = verName   #俗名
        ws.cell(row=a, column=3).value = sciName   #學名
        ws.cell(row=a, column=6).value = category     #類別
        ws.cell(row=a, column=7).value = found_url    #搜尋網址
        ws.cell(row=a, column=8).value = kingdom  #界
        ws.cell(row=a, column=9).value = phylum   #門
        ws.cell(row=a, column=10).value = _class   #綱
        ws.cell(row=a, column=11).value = order    #目
        ws.cell(row=a, column=12).value = family   #科
        ws.cell(row=a, column=13).value = genus    #屬
        ws.cell(row=a, column=14).value = occur_record_num     #物種出現紀錄次數
        ws.cell(row=a, column=15).value = occur_record_url     #物種出現紀錄網址
        ws.cell(row=a, column=16).value = nature_history_num   #自然史典藏
        ws.cell(row=a, column=17).value = nature_history_url   #自然史典藏
        a += 1

In [24]:
# 執行結果並存入txt檔
file_name = 'output_sp_result.txt'  # 替換為您的檔案名稱
head = '{ ''features'' : {\n\n'
tail = '\n} , type = ''FeatureCollection'' }'

search_list = mk_search_list(name_list)
body_list = iter_TBIA_result(search_list)
body = file_line(file_name)

output_sp_result = str( head + body + tail )
write_read_txt('output_sp_dict.txt', output_sp_result)

'{ features : {\n\n\n{\'verName\': \'紅檜\', \'sciName\': \'Chamaecyparis formosensis\', \'category\': 9, \'found_url\': \'https://tbiadata.tw/zh-hant/search/full?keyword=%E7%B4%85%E6%AA%9C\', \'taxon_rank_kingdom\': \'Plantae 植物界\', \'taxon_rank_phylum\': \'Tracheophyta 維管束植物門\', \'taxon_rank_class\': \'Pinopsida 松綱\', \'taxon_rank_order\': \'Cupressales 柏目\', \'taxon_rank_family\': \'Cupressaceae 柏科\', \'taxon_rank_genus\': \'Chamaecyparis 扁柏屬\', \'occur_record_num\': \'12465\', \'occur_record_url\': \'https://tbiadata.tw/zh-hant/search/full?keyword=%E7%B4%85%E6%AA%9C&record_type=occ&key=taxonID&value=t0052942&scientific_name=Chamaecyparis+formosensis&limit=12465&page=1&from=card&get_record=true&orderby=scientificName&sort=asc\', \'nature_history_num\': \'181\', \'nature_history_url\': \'https://tbiadata.tw/zh-hant/search/full?keyword=%E7%B4%85%E6%AA%9C&record_type=col&key=taxonID&value=t0052942&scientific_name=Chamaecyparis+formosensis&limit=181&page=1&from=card&get_record=true&orderb

In [25]:
# 將執行結果存入excel檔
txt_file_name = 'output_sp_result.txt'
sheet_title = '合歡山植物名錄 (HeHuanMt. Plant basic information)'
excel_file_name = 'sp_information.xlsx'


mk_output_excel(sheet_title, txt_file_name)
save_excel_file(excel_file_name)

C:\Users\鴻\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


7. get_ocr_nh_data(ocr_nh_url)、mk_ocr_nh_page_url(ocr_nh_num, ocr_nh_url)、get_occur_record_data(ocr_nh_num, ocr_nh_url)

In [41]:
def mk_ocr_nh_url_list(file_name):
    result_lists = mk_result_list(file_name)
    ocr_nh_url_list = []
    for result_list in enumerate(result_lists, start=2):
        try:
            # 依字典順序讀取字典中的物種出現紀錄、自然典藏史網站
            verName = result_list[1][1]
            sciName = result_list[2][1]
            occur_record_num = result_list[11][1]
            occur_record_url = result_list[12][1]
            nature_history_num = result_list[13][1]
            nature_history_url = result_list[14][1]
            list_ = [verName, sciName, occur_record_num, occur_record_url, nature_history_num, nature_history_url]
        except IndexError as e:
            continue

        ocr_nh_url_list.append(list_)
    return ocr_nh_url_list

In [42]:
def get_ocr_nh_data(ocr_nh_url):
    # 搜尋網址中的left_area、left_area
    soup = drive_TBIA_html(ocr_nh_url)
    soup = soup.find('div', class_='sc_de_main')

    left_area = soup.find('div', class_='left_area')
    left_area = left_area.find('div', class_='whcont')
    left_area = left_area.find('ul', class_='inforall')
    left_area = left_area.find_all('li')

    right_area = soup.find('div', class_='right_area')
    left_area = soup.find('ul', class_='inforall')
    right_area = right_area.find_all('li')

    # 取得left_area中的數據
    for item in left_area:
    # for i in range(len(left_area)):
    #     item = left_area[i]
        key = item.find('h3').text
        value = item.find('p').text

        if key == '入口網ID：':
            recordedByID = value
        elif key == 'occurrenceID：':
            occurrenceID = value
        elif key == '記錄者：':
            recordedBy = value
        elif key == '紀錄日期：':
            date = value
        elif key == '出現地：':
            locality = value
        else:
            continue

    # 取得left_area中的數據
    for item in right_area:
    # for j in range(len(right_area)):
    #     item = right_area[j]
        key = item.find('h3').text
        value = item.find('p').text

        if key == '經度：':
            decimalLongitude = value
        elif key == '緯度：':
            decimalLatitude = value
        elif key == '座標誤差（公尺）：':
            coordinateUncertaintyInMeters = value
        elif key == '座標模糊化程度':
            dataGeneralization = value
        else:
            continue

    time.sleep(delay)
    driver.quit()
    return recordedByID, occurrenceID, recordedBy, date, locality, decimalLongitude, decimalLatitude, coordinateUncertaintyInMeters, dataGeneralization

In [43]:
def mk_ocr_nh_item_url(ocr_nh_num, ocr_nh_url):

    # 翻頁並生產對應的網址
    item_per_page = 10
    page_num = math.ceil(ocr_nh_num/item_per_page)

    ocr_nh_item_url_list = []
    for i in range(page_num):
        ocr_nh_page_url = ocr_nh_url.split('&page=')[0] + '&page=' + str(i+1) + re.sub(r'^\d+', '', ocr_nh_url.split('&page=')[1])
        soup = drive_TBIA_html(ocr_nh_page_url)
        ocr_list = soup.find('table', class_='table_style1 record_table')
        ocr_list = ocr_list.find_all('tr')

        for i in range(len(ocr_list) -1):
            # 找出每一筆資料網址並產生網址
            ocr = ocr_list[i+1]
            ocr = ocr.find('a', target = '_blank')
            if not ocr and 'href' not in ocr.attrs:
                continue
            ocr_item_url = ocr['href']
            ocr_item_url = 'https://tbiadata.tw' + ocr_item_url
            ocr_nh_item_url_list.append(ocr_item_url)
            # print(ocr_item_url)

    return ocr_nh_item_url_list

In [44]:
def get_occur_record_data(ocr_nh_num, ocr_nh_url):
    ocr_nh_item_url_list = mk_ocr_nh_item_url(ocr_nh_num, ocr_nh_url)

    # 搜尋並獲取每一筆資料的數據
    for i in range(len(ocr_nh_item_url_list)):
        ocr_nh_item_url = ocr_nh_item_url_list[i]
        ocr_nh_data = get_ocr_nh_data(ocr_nh_item_url)

        recordedByID = str(ocr_nh_data[0])
        occurrenceID = str(ocr_nh_data[1])
        recordedBy = str(ocr_nh_data[2])
        date = str(ocr_nh_data[3])
        locality = str(ocr_nh_data[4])
        decimalLongitude,  = str(round(ocr_nh_data[5], 6))
        decimalLatitude = str(round(ocr_nh_data[6], 6))
        coordinateUncertaintyInMeters = str(ocr_nh_data[7])
        dataGeneralization = str(ocr_nh_data[8])

        results = {'recordedByID': recordedByID, 'occurrenceID': occurrenceID, 'recordedBy': recordedBy, 'date': date, 
                   'locality': locality, 'decimalLongitude': decimalLongitude, 'decimalLatitude': decimalLatitude, 'coordinateUncertaintyInMeters': coordinateUncertaintyInMeters, 'dataGeneralization': dataGeneralization}
        write_read_txt('ocr_nh_data_result.txt', results)
        yield ocr_nh_data

In [104]:
# 爬蟲逐個提取物種出現紀錄(occur_record, ocr)的資訊
ocr_nh_url_list =  mk_ocr_nh_url_list(file_name)
for ocr_nh in ocr_nh_url_list:
    verName = ocr_nh[1]
    sciName = ocr_nh[2]
    occur_record_num = ocr_nh[3]
    occur_record_url = ocr_nh[4]
    get_occur_record_data(occur_record_num, occur_record_url)

In [105]:
# 爬蟲逐個提取自然史典藏(nature_history, nh)的資訊
ocr_nh_url_list =  mk_ocr_nh_url_list(file_name)
for ocr_nh in ocr_nh_url_list:
    verName = ocr_nh[1]
    sciName = ocr_nh[2]
    nature_history_num = ocr_nh[5]
    nature_history_url = ocr_nh[6]
    get_occur_record_data(nature_history_num, nature_history_url)